# Image To Graph

[`eucare.image_to_graph`](../reference/eucare/image_to_graph.md) digitises a raster image of a line drawing (e.g. a scanned or photographed hand-drawn crease pattern) into a planar half-edge graph.

The pipeline is:

1. downsample the image,
2. threshold to a binary foreground mask,
3. close gaps with a morphological closing,
4. skeletonise and prune the result,
5. detect branch points and the edge segments between them,
6. merge branch points that are closer than a chosen cutoff,
7. emit an [`EuclideanPositionHEG`](../reference/eucare/half.md).

The function plots intermediate results so you can pick `threshold` and `edge_length_cutoff` interactively.

Requires the optional dependencies `scikit-image` and `mahotas`.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)

import eucare as ec
from eucare.image_to_graph import image_to_graph

## Convert the image

We use the photograph `test_graph_image.jpg` shipped under `misc_outputs/`. The two parameters that almost always need tuning are:

- `threshold` — grayscale cutoff separating ink from paper. After the initial run the function shows you the grayscale histogram so you can pick a value.
- `edge_length_cutoff` — distance below which neighbouring branch-point clusters are merged into a single vertex. The function shows you the distribution of edge lengths to help pick this.

The function prints/plots the original image, the grayscale conversion, the thresholded foreground, the morphological closing, and finally the labelled edges.

In [ ]:
G = image_to_graph(
    'images/test_graph_image.jpg',
    threshold=75,
    closing_iterations=3,
    edge_length_cutoff=40,
)

## Inspect the resulting graph

`image_to_graph` returns a fully-fledged [`EuclideanPositionHEG`](../reference/eucare/half.md) — the same data structure produced by the tile-growth pipeline. Lengths and angles have already been recomputed from the extracted vertex positions, so it is ready for downstream processing (Conway operators, reciprocal figures, folding, etc.).

In [ ]:
print(f'{len(G.vertices)} vertices, {len(G.halfedges) // 2} edges, {len(G.faces)} faces')
G.show()

## Saving and loading

Digitising a drawing is comparatively expensive, so you typically want to persist the result. Eucare's native YAML format `.heg` is round-trippable via [`ec.io.save_graph`](../reference/eucare/io.md) and [`ec.io.load_graph`](../reference/eucare/io.md):

In [ ]:
# ec.io.save_graph('graphs/test_graph_image.heg', G, overwrite=True)
# G_loaded = ec.io.load_graph('graphs/test_graph_image.heg')
# G_loaded.show()